# Ollama + Qwen2.5 14B on Colab (GPU)
Run each cell in order. After the last cell, copy the ngrok URL into your local `server/.env`.

In [1]:
# Cell 1 — Verify GPU is available
!nvidia-smi

Wed May 13 08:22:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Install dependencies, Ollama, and pyngrok
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok -q
print('Install complete.')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Install complete.


In [3]:
# Cell 3 — Start Ollama server in the background
import subprocess, time, os, shutil, urllib.request

# Refresh PATH so Python finds the ollama binary installed in Cell 2
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
print('Ollama binary:', ollama_bin)

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'

proc = subprocess.Popen(
    [ollama_bin, 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)

# Poll until the server is actually accepting connections (up to 60s)
print('Waiting for Ollama server to be ready...')
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/', timeout=2)
        print(f'Ollama server ready after {i+1}s (PID {proc.pid})')
        break
    except Exception:
        time.sleep(1)
else:
    print('WARNING: Ollama server did not respond within 60s — check for errors')

Ollama binary: /usr/local/bin/ollama
Waiting for Ollama server to be ready...
Ollama server ready after 1s (PID 14130)


In [4]:
# Cell 4 — Pull qwen2.5:14b (~9 GB, fits in T4's 15 GB VRAM)
# This model is significantly better than mistral:7b at structured JSON output.
# Download takes ~5-10 min on Colab.
!ollama pull qwen2.5:14b

In [5]:
# Cell 5 — Quick smoke test: confirm the model responds
import urllib.request, json

payload = json.dumps({
    'model': 'qwen2.5:14b',
    'messages': [{'role': 'user', 'content': 'Reply with just the word: READY'}],
    'stream': False
}).encode()

req = urllib.request.Request(
    'http://localhost:11434/api/chat',
    data=payload,
    headers={'Content-Type': 'application/json'}
)

# First request loads the model into VRAM — allow up to 3 minutes
print('Sending test prompt (first load may take up to 3 minutes)...')
with urllib.request.urlopen(req, timeout=180) as r:
    result = json.load(r)
print('Model response:', result['message']['content'])
print('Model is working correctly.')

Sending test prompt (first load may take up to 3 minutes)...
Model response: READY
Model is working correctly.


In [6]:
# Cell 6 — Expose Ollama via ngrok
# Auth token:  https://dashboard.ngrok.com/get-started/your-authtoken
# API key:     https://dashboard.ngrok.com/api-keys  (needed for instant endpoint release)
NGROK_TOKEN   = '3BRe0cL8Qf8pB9PvbC9Qhd48A2r_4KyfysVT3EWq6tudYbvGA'  # <-- replace
NGROK_API_KEY = 'ak_3Df1n2JZ2ehyj1u3VSyqvUqOjZp'  # <-- paste your API key here for instant endpoint release (optional)

import time, json, urllib.request, urllib.error
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

# Kill any locally-visible tunnels and the local agent
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass
ngrok.kill()
time.sleep(3)

# If an API key is provided, use the ngrok REST API to immediately delete any
# cloud-side endpoints — this avoids the 30–180s cloud expiry wait entirely.
if NGROK_API_KEY:
    print('Attempting API-based endpoint cleanup...')
    try:
        list_req = urllib.request.Request(
            'https://api.ngrok.com/endpoints',
            headers={'Authorization': f'Bearer {NGROK_API_KEY}',
                     'Ngrok-Version': '2'}
        )
        with urllib.request.urlopen(list_req, timeout=10) as r:
            data = json.load(r)
            endpoints = data.get('endpoints', [])
        print(f'Found {len(endpoints)} cloud endpoint(s).')
        for ep in endpoints:
            print(f'  Deleting: {ep["public_url"]} (id={ep["id"]})')
            del_req = urllib.request.Request(
                f'https://api.ngrok.com/endpoints/{ep["id"]}',
                method='DELETE',
                headers={'Authorization': f'Bearer {NGROK_API_KEY}',
                         'Ngrok-Version': '2'}
            )
            urllib.request.urlopen(del_req, timeout=10)
            print(f'  Deleted.')
        if endpoints:
            time.sleep(3)
        else:
            print('  No endpoints to delete (may already be released or API key lacks access).')
    except urllib.error.HTTPError as ex:
        body = ex.read().decode()
        print(f'API cleanup FAILED — HTTP {ex.code}: {body}')
        print('Check your NGROK_API_KEY is correct and has sufficient permissions.')
    except Exception as ex:
        print(f'API cleanup FAILED — {ex}')

# Connect — retry with backoff as fallback if no API key or deletion was slow
print('Connecting ngrok tunnel...')
tunnel = None
for attempt in range(1, 11):  # up to 10 × 20s = ~200s
    try:
        tunnel = ngrok.connect(11434, 'http')
        print(f'Tunnel established on attempt {attempt}.')
        break
    except Exception as e:
        if 'ERR_NGROK_334' in str(e) or 'already online' in str(e):
            print(f'Attempt {attempt}/10: endpoint still reserved, waiting 20s...')
            ngrok.kill()
            time.sleep(20)
        else:
            raise

if tunnel is None:
    raise RuntimeError(
        'Could not claim the ngrok endpoint after 10 attempts.\n'
        'Delete it manually at: https://dashboard.ngrok.com/cloud-edge/endpoints\n'
        'Then re-run this cell.'
    )

ollama_url = tunnel.public_url
print('=' * 60)
print('Ollama is live at:', ollama_url)
print('=' * 60)
print()
print('Paste this into your local server/.env:')
print(f'OLLAMA_BASE_URL={ollama_url}')
print('OLLAMA_CONCURRENCY=3')


Attempting API-based endpoint cleanup...
API cleanup FAILED — HTTP 400: {"error_code":"ERR_NGROK_208","status_code":400,"msg":"The authentication you specified is actually an API key ID, not an API key token. Your credential: 'ak_3Df1n2JZ2ehyj1u3VSyqvUqOjZp'. Check your records for an API key with the form FIRSTPART_SECONDPART. API keys and instructions are available on your dashboard: https://dashboard.ngrok.com/api-keys","details":{"operation_id":"op_3Df64A48QTTDmCqKscRjw6Q47Db"}}
Check your NGROK_API_KEY is correct and has sufficient permissions.
Connecting ngrok tunnel...
Tunnel established on attempt 1.
Ollama is live at: https://unadmonished-maegan-proscientific.ngrok-free.dev

Paste this into your local server/.env:
OLLAMA_BASE_URL=https://unadmonished-maegan-proscientific.ngrok-free.dev
OLLAMA_CONCURRENCY=3


In [ ]:
# Cell 7 — Keep-alive: run this to prevent Colab from timing out
# Leave this cell running while you grade. Stop it when done.
import time
print('Keep-alive running. Stop this cell when grading is complete.')
while True:
    time.sleep(30)
    # Ping Ollama every 30 seconds to keep it warm
    try:
        urllib.request.urlopen('http://localhost:11434/', timeout=5)
        print('.', end='', flush=True)
    except:
        print('x', end='', flush=True)

Keep-alive running. Stop this cell when grading is complete.
...........................................................................................................